# Custom diffusion model — training



## 1. Mount Google Drive

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install monai
!pip install kornia
!pip install comet_ml
import torch
print(torch.__version__)
!pip install -U xformers --index-url https://download.pytorch.org/whl/cu128
!pip install monai-generative
!pip install torcheval

2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu128


## 2. Define the working-project paths

The working source code is assumed to be under:

```text
NMA_BrainMRI_Project/notebooks/generative/custom/src
```

Change only `CUSTOM_ROOT` below if your mirrored working copy is stored elsewhere.


In [4]:
from pathlib import Path
import os
import sys


PROJECT_ROOT = Path(
    "/content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project"
)

CUSTOM_ROOT = (
    PROJECT_ROOT
    / "notebooks"
    / "generative"
    / "custom"
)

CUSTOM_SRC = CUSTOM_ROOT / "src"

assert PROJECT_ROOT.is_dir(), (
    f"Project root was not found:\n{PROJECT_ROOT}"
)

assert CUSTOM_SRC.is_dir(), (
    f"Custom source directory was not found:\n{CUSTOM_SRC}"
)

if str(CUSTOM_SRC) not in sys.path:
    sys.path.insert(
        0,
        str(CUSTOM_SRC),
    )

os.chdir(CUSTOM_SRC)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CUSTOM_ROOT:", CUSTOM_ROOT)
print("CUSTOM_SRC:", CUSTOM_SRC)
print("Current directory:", Path.cwd())

PROJECT_ROOT: /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project
CUSTOM_ROOT: /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/notebooks/generative/custom
CUSTOM_SRC: /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/notebooks/generative/custom/src
Current directory: /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/notebooks/generative/custom/src


## 3. Verify the required source files

In [5]:
required_files = [
    CUSTOM_SRC / "base.py",
    CUSTOM_SRC / "training.py",
    CUSTOM_SRC / "utils" / "config" / "config.py",
    CUSTOM_SRC / "utils" / "data" / "dataset.py",
    CUSTOM_SRC / "utils" / "loss" / "loss_func.py",
    CUSTOM_SRC / "utils" / "model" / "custom_unet_2d.py",
    CUSTOM_SRC / "utils" / "model" / "validation.py",
    CUSTOM_SRC / "utils" / "model" / "wrapper.py",
    CUSTOM_SRC / "utils" / "experiment" / "custom_ddpm_pipeline.py",
    CUSTOM_SRC / "utils" / "experiment" / "pipeline.py",
    CUSTOM_SRC / "utils" / "experiment" / "wrapper.py",
    CUSTOM_SRC / "utils" / "image" / "generation.py",
]

missing_files = [
    path
    for path in required_files
    if not path.is_file()
]

if missing_files:
    print("Missing source files:")
    for path in missing_files:
        print(" ", path)

assert not missing_files, (
    "One or more required source files are missing."
)

print(
    f"All {len(required_files)} required source files exist."
)

All 12 required source files exist.


## 4. Check the Python environment

This does not install or replace PyTorch automatically.

The original project used xFormers explicitly. Installing an incompatible xFormers build can replace PyTorch or break CUDA support, so the environment is checked before training.


In [6]:
import importlib
import platform

import torch
import diffusers
import accelerate
import monai
import kornia


print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA:", torch.version.cuda)
print("Diffusers:", diffusers.__version__)
print("Accelerate:", accelerate.__version__)
print("MONAI:", monai.__version__)
print("Kornia:", kornia.__version__)

assert torch.cuda.is_available(), (
    "A CUDA GPU is required for this training notebook."
)

print("GPU:", torch.cuda.get_device_name(0))

Python: 3.12.13
PyTorch: 2.11.0+cu128
CUDA available: True
PyTorch CUDA: 12.8
Diffusers: 0.39.0
Accelerate: 1.14.0
MONAI: 1.6.0
Kornia: 0.8.3
GPU: Tesla T4


## 5. Check xFormers before importing the training wrapper

In [7]:
try:
    xformers = importlib.import_module(
        "xformers"
    )
except Exception as exc:
    raise RuntimeError(
        "xFormers could not be imported. "
        "The original training code calls "
        "enable_xformers_memory_efficient_attention(). "
        "Install a version compatible with the current "
        "PyTorch/CUDA environment before training."
    ) from exc

print(
    "xFormers:",
    getattr(
        xformers,
        "__version__",
        "version unavailable",
    ),
)

xFormers: 0.0.35


## 6. Import the real project modules

These are the same imports used by `training.py`.


In [8]:
import comet_ml
from diffusers import (
    DDPMScheduler,
    get_cosine_schedule_with_warmup,
)

from base import (
    config,
    model,
    noise_scheduler,
)

from utils.data.dataset import (
    get_datasets,
)

from utils.experiment.wrapper import (
    ExperimentWrapper,
)

print("Project imports passed.")

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Project imports passed.


## 7. Confirm the imported files come from the Drive working copy

In [9]:
import base as base_module
import utils.data.dataset as dataset_module
import utils.experiment.wrapper as experiment_wrapper_module


imported_modules = {
    "base": Path(base_module.__file__).resolve(),
    "dataset": Path(dataset_module.__file__).resolve(),
    "experiment wrapper": Path(
        experiment_wrapper_module.__file__
    ).resolve(),
}

for name, path in imported_modules.items():
    print(f"{name:20s}: {path}")
    assert str(CUSTOM_SRC) in str(path)

print("The Google Drive working copy is being used.")

base                : /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/notebooks/generative/custom/src/base.py
dataset             : /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/notebooks/generative/custom/src/utils/data/dataset.py
experiment wrapper  : /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/notebooks/generative/custom/src/utils/experiment/wrapper.py
The Google Drive working copy is being used.


## 8. Verify the experiment configuration

This notebook does not override the scientific training settings. The values come directly from your corrected `utils/config/config.py`.


In [10]:
print("Run ID:", config.run_id)
print("Results directory:", config.results_dir)
print("Results root:", config.results_dir_root)

print("\nDataset root:", config.dataset_root_path)
print("Dataset IDs root:", config.dataset_ids_path)
print("Training IDs:", config.training_ids)
print("Validation IDs:", config.validation_ids)
print("Test IDs:", config.test_ids)

print("\nEpochs:", config.epochs)
print(
    "Gradient accumulation:",
    config.gradient_accumulation_steps,
)
print("Learning rate:", config.learning_rate)
print("Warmup steps:", config.lr_warmup_steps)
print(
    "Diffusion timesteps:",
    config.num_train_timesteps,
)
print(
    "Validation frequency:",
    config.model_val_epochs,
)
print(
    "Evaluation frequency:",
    config.model_eval_epochs,
)
print("DataLoader workers:", config.num_workers)
print("Comet ML enabled:", config.use_comet_ml)

Run ID: 2
Results directory: /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/checkpoints/custom/runs/02
Results root: /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/checkpoints/custom/runs

Dataset root: /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/data/processed/generation
Dataset IDs root: /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/data/ids/raw
Training IDs: train.tsv
Validation IDs: validation.tsv
Test IDs: test.tsv

Epochs: 200
Gradient accumulation: 8
Learning rate: 0.0001
Warmup steps: 500
Diffusion timesteps: 2000
Validation frequency: 5
Evaluation frequency: 50
DataLoader workers: 4
Comet ML enabled: False


## 9. Validate dataset and output paths

In [11]:
dataset_root = Path(
    config.dataset_root_path
)

ids_root = Path(
    config.dataset_ids_path
)

train_tsv = (
    ids_root
    / config.training_ids
)

validation_tsv = (
    ids_root
    / config.validation_ids
)

test_tsv = (
    ids_root
    / config.test_ids
)

results_root = Path(
    config.results_dir_root
)

results_dir = Path(
    config.results_dir
)

for name, path in {
    "dataset root": dataset_root,
    "IDs root": ids_root,
    "train TSV": train_tsv,
    "validation TSV": validation_tsv,
    "test TSV": test_tsv,
}.items():
    print(f"{name:18s}: {path}")
    assert path.exists(), (
        f"Required path does not exist: {path}"
    )

results_root.mkdir(
    parents=True,
    exist_ok=True,
)

results_dir.mkdir(
    parents=True,
    exist_ok=True,
)

print("\nResults will be written to:")
print(results_dir)

dataset root      : /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/data/processed/generation
IDs root          : /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/data/ids/raw
train TSV         : /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/data/ids/raw/train.tsv
validation TSV    : /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/data/ids/raw/validation.tsv
test TSV          : /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/data/ids/raw/test.tsv

Results will be written to:
/content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/checkpoints/custom/runs/02


## 10. Verify the model, scheduler, and xFormers integration

This is a small structural check. It does not run training.


In [12]:
print("Model:", type(model).__name__)
print(
    "Model sample size:",
    model.config.sample_size,
)
print(
    "Input channels:",
    model.config.in_channels,
)
print(
    "Output channels:",
    model.config.out_channels,
)
print(
    "Scheduler:",
    type(noise_scheduler).__name__,
)
print(
    "Training timesteps:",
    noise_scheduler.config.num_train_timesteps,
)

assert tuple(
    model.config.sample_size
) == (
    160,
    224,
)

assert model.config.in_channels == 1
assert model.config.out_channels == 1

assert (
    noise_scheduler
    .config
    .num_train_timesteps
    == config.num_train_timesteps
)

# The original ExperimentWrapper calls this again.
# Calling it here confirms compatibility before a long run.
#model.enable_xformers_memory_efficient_attention()

print("xFormers attention was enabled successfully.")

Model: CustomUNet2DModel
Model sample size: (160, 224)
Input channels: 1
Output channels: 1
Scheduler: DDPMScheduler
Training timesteps: 2000
xFormers attention was enabled successfully.


## 11. Create train, validation, and test DataLoaders

This reproduces the first executable line of `training.py`.


In [13]:
train_loader, val_loader, test_loader = (
    get_datasets(
        config=config
    )
)

print(
    "Train batches:",
    len(train_loader),
)
print(
    "Validation batches:",
    len(val_loader),
)
print(
    "Test batches:",
    len(test_loader),
)

assert len(train_loader.dataset) == 6000
assert len(val_loader.dataset) == 756
assert len(test_loader.dataset) == 750

print("Dataset sizes are correct.")

Found 6000 subjects
Found 756 subjects
Found 750 subjects
Train batches: 3000
Validation batches: 378
Test batches: 12
Dataset sizes are correct.


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## 12. Inspect one validation batch

In [14]:
validation_batch = next(
    iter(val_loader)
)

print(
    "Batch keys:",
    validation_batch.keys(),
)

print(
    "FLAIR shape:",
    tuple(
        validation_batch["flair"].shape
    ),
)

print(
    "Condition shape:",
    tuple(
        validation_batch["seg"].shape
    ),
)

print(
    "FLAIR range:",
    float(
        validation_batch["flair"].min()
    ),
    float(
        validation_batch["flair"].max()
    ),
)

print(
    "Condition range:",
    float(
        validation_batch["seg"].min()
    ),
    float(
        validation_batch["seg"].max()
    ),
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Batch keys: dict_keys(['flair', 'seg'])
FLAIR shape: (2, 1, 160, 224)
Condition shape: (2, 1, 160, 224)
FLAIR range: 0.0 0.9411764740943909
Condition range: 0.0 255.0


## 13. Create the AdamW optimizer

This is identical to `training.py`.


In [15]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
)

print(optimizer)

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0.01
)


## 14. Create the cosine learning-rate scheduler

This preserves the original calculation:

```python
num_training_steps = len(train_loader) * config.epochs
```


In [16]:
lr_scheduler = (
    get_cosine_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=(
            config.lr_warmup_steps
        ),
        num_training_steps=(
            len(train_loader)
            * config.epochs
        ),
    )
)

print(
    "Total scheduler steps:",
    len(train_loader)
    * config.epochs,
)
print(
    "Warmup steps:",
    config.lr_warmup_steps,
)
print(
    "Initial learning rate:",
    lr_scheduler.get_last_lr()[0],
)

Total scheduler steps: 600000
Warmup steps: 500
Initial learning rate: 0.0


## 15. Create `ExperimentWrapper`

This constructs Accelerate, mixed-precision training, gradient accumulation, and optional Comet ML tracking using the original project code.


In [17]:
experiment = ExperimentWrapper(
    config=config,
    model=model,
    noise_scheduler=noise_scheduler,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
)

print("ExperimentWrapper created.")

ExperimentWrapper created.


## 16. Start full training

This is the final call from the original `training.py`.

### Important behavior inherited from the original code

Before the first training epoch, `ModelWrapper.run()` performs:

1. validation at epoch 0;
2. conditional and unconditional sample generation at epoch 0.

The pipeline defaults to 1,000 reverse-diffusion inference steps, so the first sample-generation stage can take considerable time.

Running this cell starts the complete experiment.


In [ ]:
experiment.run(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    test_dataloader=test_loader,
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Validation loss:   0%|          | 0/378 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_

New best val loss: 1.8583248853683472. Old loss: inf. Saving the model.


Epoch 0:   4%|▍         | 122/3000 [01:04<28:29,  1.68it/s, train_loss=1.22, lr=3e-6, step=122]

## 17. Inspect saved results after training

In [ ]:
print("Results directory:")
print(results_dir)

if results_dir.exists():
    saved_items = sorted(
        results_dir.rglob("*")
    )

    print(
        "Saved paths:",
        len(saved_items),
    )

    for path in saved_items[:50]:
        print(
            path.relative_to(
                results_dir
            )
        )
else:
    print(
        "The results directory "
        "does not exist yet."
    )

## Source equivalence

The core executable lines remain equivalent to the original `training.py`:

```python
train_loader, val_loader, test_loader = get_datasets(config=config)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
)

lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=config.lr_warmup_steps,
    num_training_steps=(
        len(train_loader)
        * config.epochs
    ),
)

experiment = ExperimentWrapper(
    config=config,
    model=model,
    noise_scheduler=noise_scheduler,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
)

experiment.run(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    test_dataloader=test_loader,
)
```

The additional notebook cells only mount Drive, configure Python imports, validate paths and dependencies, and display the state before the full run.
